[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/06_INE.ipynb)

# 06 — INE: OCR + NER (spaCy)

Imagen `data/fakeine.jpeg` → texto (EasyOCR o Azure abajo) → entidades con `es_core_news_sm`.

In [ ]:
# %pip install -q easyocr spacy
# !python -m spacy download es_core_news_sm

from pathlib import Path

import easyocr
import numpy as np
import spacy
from IPython.display import HTML, display
from PIL import Image, ImageOps
from spacy import displacy

RUTA_INE = Path("data/fakeine.jpeg")
ocr = easyocr.Reader(["es"], gpu=False)
nlp = spacy.load("es_core_news_sm")

In [ ]:
def ine_imagen_a_texto(ruta, lector_ocr):
    img = Image.open(ruta).convert("RGB")
    img = ImageOps.exif_transpose(img)
    w, h = img.size
    m = max(w, h)
    if m > 2000:
        s = 2000 / m
        img = img.resize((int(w * s), int(h * s)), Image.LANCZOS)
    tiras = lector_ocr.readtext(np.asarray(img), detail=0)
    return "\n".join(tiras)


def texto_a_entidades_visual(texto, modelo_nlp):
    doc = modelo_nlp(texto[:100_000])
    display(HTML(displacy.render(doc, style="ent", jupyter=False)))
    return [(e.text.strip(), e.label_) for e in doc.ents if e.text.strip()]

In [ ]:
texto = ine_imagen_a_texto(RUTA_INE, ocr)
print(texto)
lista = texto_a_entidades_visual(texto, nlp)
lista

## (Opcional) **Document Intelligence** + mismo NER

`prebuilt-read` en la nube. En `.env`: `AZURE_DI_ENDPOINT` y clave (`AZURE_FOUNDRY_API` o `AZURE_DI_KEY`). Alternativa: `AZURE_DI_NOMBRE_RECURSO` si no pones la URL entera.

In [ ]:
# %pip install -q azure-ai-documentintelligence python-dotenv
import os
from pathlib import Path

from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

# "prebuilt-read" | "prebuilt-layout" | "prebuilt-idDocument"
MODELO_DI = "prebuilt-idDocument"


def _texto_desde_resultado(res) -> str:
    t = (getattr(res, "content", None) or "").strip()
    if t:
        return t
    if res.paragraphs:
        return " ".join(p.content for p in res.paragraphs if p.content)
    # idDocument / layout: a veces el texto salta más en fields
    lineas = []
    for doc in res.documents or []:
        campos = getattr(doc, "fields", None) or {}
        for c in (campos.values() if isinstance(campos, dict) else []):
            if c is None:
                continue
            s = (getattr(c, "content", None) or getattr(c, "value", None) or "")
            s = s if isinstance(s, str) else (str(s) if s is not None else "")
            if s.strip():
                lineas.append(s.strip())
    return "\n".join(lineas) if lineas else ""


def _endpoint_document_intelligence() -> str:
    ep = (os.getenv("AZURE_DI_ENDPOINT") or "").strip()
    if ep:
        return ep.rstrip("/") + "/"
    nom = (os.getenv("AZURE_DI_NOMBRE_RECURSO") or "").strip()
    if nom:
        return f"https://{nom}.cognitiveservices.azure.com/"
    return ""


def ine_imagen_a_texto_azure(ruta) -> str:
    ep = _endpoint_document_intelligence()
    if not ep:
        raise ValueError("Falta AZURE_DI_ENDPOINT o AZURE_DI_NOMBRE_RECURSO en .env")
    key = (os.getenv("AZURE_DI_KEY") or os.getenv("AZURE_FOUNDRY_API") or "").strip()
    if not key:
        raise ValueError("Falta AZURE_FOUNDRY_API o AZURE_DI_KEY")

    client = DocumentIntelligenceClient(
        endpoint=ep,
        credential=AzureKeyCredential(key),
    )
    modelo = "prebuilt-read"
    # modelo = "prebuilt-layout"
    with open(ruta, "rb") as f:
        poller = client.begin_analyze_document(
            modelo, f, content_type="application/octet-stream", locale="es-MX"
        )
    res = poller.result()
    return (getattr(res, "content", None) or "").strip() or " ".join(
        p.content for p in (res.paragraphs or []) if p.content
    )

In [ ]:
texto_azure = ine_imagen_a_texto_azure(Path("data/fakeine.jpeg"))
print(texto_azure)
texto_a_entidades_visual(texto_azure, nlp)